# Task 2 — Data Discovery, Profiling and Cleaning
**Project:** SkyPrint — Flight Tracking & Climate Impact Analysis

Inputs (from Task 1, `data/raw/`):
- latest `opensky_<timestamp>.json` — OpenSky state vectors from the Saudi-area extraction
- latest `openmeteo_<timestamp>.json` — weather for aircraft-relevant grid locations and pressure levels
- `aircraftDatabase.csv` — aircraft lookup (`plane_id` / ICAO24 → aircraft metadata and `typecode`)

This notebook keeps the same Task 2 methodology:
**Load → Flatten → Profile → Document issues → Clean → Save**

Outputs:
- `data/interim/cleaned_opensky.csv`
- `data/interim/cleaned_weather.csv`
- `data/interim/cleaned_aircraft.csv`


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Resolve project paths safely
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', None)

def profile(df):
    rows = []
    for c in df.columns:
        s = df[c]
        sample = s.dropna().iloc[0] if s.notna().any() else None
        rows.append({
            'column': c,
            'dtype': str(s.dtype),
            'null_count': int(s.isna().sum()),
            'percent_null': round(100 * s.isna().sum() / len(s), 2) if len(s) else 0,
            'unique_count': int(s.nunique(dropna=True)),
            'sample_value': sample
        })
    return pd.DataFrame(rows)

def latest_timestamped_file(prefix):
    files = list(RAW_DIR.glob(f'{prefix}_*.json'))
    if not files:
        fallback = RAW_DIR / f'{prefix}.json'
        if fallback.exists():
            return fallback
        raise FileNotFoundError(f'No {prefix} JSON file found in {RAW_DIR}')
    return max(files, key=lambda p: p.stat().st_mtime)


## 1. OpenSky — Load and Flatten

`/states/all` returns a JSON object with `time` and a nested `states` list.
Task 1 uses `extended=1`, so each state vector has 18 fields including `category`.


In [2]:
opensky_path = latest_timestamped_file('opensky')

with open(opensky_path, encoding='utf-8') as f:
    opensky_raw = json.load(f)

print('loaded file:', opensky_path.name)
print('snapshot unix time:', opensky_raw.get('time'))
print('number of state vectors:', len(opensky_raw.get('states', [])))

if opensky_raw.get('states'):
    print('fields in first state vector:', len(opensky_raw['states'][0]))
    display(opensky_raw['states'][0])


loaded file: opensky_2026-09-18_15-00-19.json
snapshot unix time: 1789744836
number of state vectors: 131
fields in first state vector: 18


['739222',
 '4XCDE   ',
 'Israel',
 1789744835,
 1789744835,
 35.0522,
 32.8457,
 426.72,
 False,
 43.76,
 17.8,
 3.25,
 None,
 449.58,
 None,
 False,
 0,
 0]

In [3]:
OPENSKY_COLUMNS = [
    'plane_id', 'flight_id', 'origin_country', 'time_position', 'last_contact',
    'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity',
    'true_track', 'vertical_rate', 'sensors', 'geo_altitude', 'squawk',
    'spi', 'source_type', 'category'
]

state_lengths = pd.Series([len(row) for row in opensky_raw.get('states', [])]).value_counts().sort_index()
print('state-vector lengths:')
print(state_lengths)

if len(state_lengths) and not (len(state_lengths) == 1 and state_lengths.index[0] == 18):
    raise ValueError('Unexpected OpenSky state-vector length. Expected 18 fields because extended=1 is used.')

df_sky = pd.DataFrame(opensky_raw.get('states', []), columns=OPENSKY_COLUMNS)
print('shape:', df_sky.shape)
df_sky.head()


state-vector lengths:
18    131
Name: count, dtype: int64
shape: (131, 18)


,plane_id,flight_id,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,source_type,category
0,739222,4XCDE,Israel,1789744835,1789744835,35.0522,32.8457,426.72,False,43.76,17.80,3.25,None,449.58,NaN,False,0,0
1,74282d,JAV271,Jordan,1789744835,1789744835,35.0496,32.0513,6019.80,False,209.04,299.97,6.18,None,6393.18,5614,False,0,0
2,8015c2,IGO094,India,1789744834,1789744835,51.5766,22.9726,10668.00,False,232.84,99.80,0.00,None,11414.76,4525,False,0,0
3,0180a0,BNL741,Libyan Arab Jamahiriya,1789744834,1789744835,50.2697,23.6820,10972.80,False,248.48,292.26,0.00,None,11711.94,0525,False,0,0
4,728679,IAW163,Iraq,1789744725,1789744725,36.0761,31.7282,990.60,False,74.17,260.02,-3.90,None,1059.18,NaN,False,0,0


## 2. OpenSky — Profiling


In [4]:
profile_sky = profile(df_sky)
profile_sky


,column,dtype,null_count,percent_null,unique_count,sample_value
0,plane_id,str,0,0.00,131,739222
1,flight_id,str,0,0.00,131,4XCDE
2,origin_country,str,0,0.00,28,Israel
3,time_position,int64,0,0.00,39,1789744835
4,last_contact,int64,0,0.00,33,1789744835
5,longitude,float64,0,0.00,131,35.0522
6,latitude,float64,0,0.00,131,32.8457
7,baro_altitude,float64,4,3.05,91,426.72
8,on_ground,bool,0,0.00,2,False
9,velocity,float64,0,0.00,131,43.76


In [5]:
print('rows:', len(df_sky), ' columns:', df_sky.shape[1])
print('exact duplicate rows:', df_sky.duplicated().sum())
print('duplicate (plane_id, time_position) pairs:',
      df_sky.duplicated(subset=['plane_id', 'time_position']).sum())

flight_text = df_sky['flight_id'].astype('string')
print('empty-string flight_ids:', (flight_text.str.strip() == '').sum())
print('missing positions:', df_sky[['latitude', 'longitude']].isna().any(axis=1).sum())
print('category value counts:')
print(df_sky['category'].value_counts(dropna=False).sort_index())


rows: 131  columns: 18
exact duplicate rows: 0
duplicate (plane_id, time_position) pairs: 0
empty-string flight_ids: 1
missing positions: 0
category value counts:
category
0    124
1      4
2      1
6      2
Name: count, dtype: int64


### Issues found — OpenSky

| # | Issue | Decision |
|---|---|---|
| 1 | `sensors` is not useful for this project and is commonly null in this endpoint | Drop |
| 2 | `flight_id` may contain fixed-width trailing spaces or empty strings | Strip whitespace; empty → missing |
| 3 | Position/time/altitude fields can legitimately be missing in a live state vector | Keep missing values; do not invent/impute flight data |
| 4 | `squawk` and `spi` are operational fields not needed for SkyPrint fuel/trajectory analysis | Drop from cleaned analytical file; raw JSON remains unchanged |
| 5 | Unix timestamps are not analysis-friendly | Convert to UTC datetime |
| 6 | Future repeated pulls can create duplicate aircraft observations | Deduplicate on (`plane_id`, `time_position`) |
| 7 | `plane_id` is the ICAO24 join key | Normalize to lowercase and trim whitespace |
| 8 | `category` is available because Task 1 uses `extended=1` | Keep |


## 3. OpenSky — Cleaning


In [6]:
df_sky_clean = df_sky.copy()

# 1) Remove columns outside the analytical scope.
df_sky_clean = df_sky_clean.drop(columns=['sensors', 'squawk', 'spi'])

# 2) Standardize identifiers/text.
df_sky_clean['plane_id'] = (
    df_sky_clean['plane_id'].astype('string').str.strip().str.lower()
)
df_sky_clean['flight_id'] = (
    df_sky_clean['flight_id'].astype('string').str.strip().replace('', pd.NA)
)
df_sky_clean['origin_country'] = (
    df_sky_clean['origin_country'].astype('string').str.strip().replace('', pd.NA)
)

# 3) Convert Unix epoch seconds to UTC datetimes.
for col in ['time_position', 'last_contact']:
    df_sky_clean[col] = pd.to_datetime(
        df_sky_clean[col], unit='s', utc=True, errors='coerce'
    )

# 4) Explicit dtypes.
df_sky_clean['on_ground'] = df_sky_clean['on_ground'].astype('boolean')
df_sky_clean['category'] = pd.to_numeric(df_sky_clean['category'], errors='coerce').astype('Int64')
df_sky_clean['source_type'] = pd.to_numeric(df_sky_clean['source_type'], errors='coerce').astype('Int64')

# 5) Deduplicate the observation natural key.
before = len(df_sky_clean)
df_sky_clean = df_sky_clean.drop_duplicates(subset=['plane_id', 'time_position'])
print(f'dropped {before - len(df_sky_clean)} duplicate OpenSky observations')

df_sky_clean.dtypes


dropped 0 duplicate OpenSky observations


plane_id                      string
flight_id                     string
origin_country                string
time_position     datetime64[s, UTC]
last_contact      datetime64[s, UTC]
longitude                    float64
latitude                     float64
baro_altitude                float64
on_ground                    boolean
velocity                     float64
true_track                   float64
vertical_rate                float64
geo_altitude                 float64
source_type                    Int64
category                       Int64
dtype: object

In [7]:
df_sky_clean.head()


,plane_id,flight_id,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,source_type,category
0,739222,4XCDE,Israel,2026-09-18 15:20:35+00:00,2026-09-18 15:20:35+00:00,35.0522,32.8457,426.72,False,43.76,17.80,3.25,449.58,0,0
1,74282d,JAV271,Jordan,2026-09-18 15:20:35+00:00,2026-09-18 15:20:35+00:00,35.0496,32.0513,6019.80,False,209.04,299.97,6.18,6393.18,0,0
2,8015c2,IGO094,India,2026-09-18 15:20:34+00:00,2026-09-18 15:20:35+00:00,51.5766,22.9726,10668.00,False,232.84,99.80,0.00,11414.76,0,0
3,0180a0,BNL741,Libyan Arab Jamahiriya,2026-09-18 15:20:34+00:00,2026-09-18 15:20:35+00:00,50.2697,23.6820,10972.80,False,248.48,292.26,0.00,11711.94,0,0
4,728679,IAW163,Iraq,2026-09-18 15:18:45+00:00,2026-09-18 15:18:45+00:00,36.0761,31.7282,990.60,False,74.17,260.02,-3.90,1059.18,0,0


In [8]:
sky_output = INTERIM_DIR / 'cleaned_opensky.csv'
df_sky_clean.to_csv(sky_output, index=False)
print('saved:', sky_output.resolve())


saved: C:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\interim\cleaned_opensky.csv


## 4. Open-Meteo — Load and Flatten

The new extraction can request **multiple aircraft-relevant locations** in one call.
Open-Meteo therefore returns either:
- one forecast object (single location), or
- a list of forecast objects (multiple locations).

Each forecast object contains hourly parallel arrays. We flatten every location and preserve its latitude/longitude.


In [9]:
meteo_path = latest_timestamped_file('openmeteo')

with open(meteo_path, encoding='utf-8') as f:
    meteo_raw = json.load(f)

print('loaded file:', meteo_path.name)
print('raw response type:', type(meteo_raw).__name__)

meteo_locations = meteo_raw if isinstance(meteo_raw, list) else [meteo_raw]
print('weather locations returned:', len(meteo_locations))


loaded file: openmeteo_2026-09-18_15-00-19.json
raw response type: list
weather locations returned: 121


In [10]:
weather_frames = []

for location in meteo_locations:
    hourly = location.get('hourly', {})
    if not hourly or 'time' not in hourly:
        continue

    frame = pd.DataFrame(hourly)
    frame['latitude'] = location.get('latitude')
    frame['longitude'] = location.get('longitude')
    frame['elevation_m'] = location.get('elevation')
    frame['location_id'] = location.get('location_id', 0)
    frame['timezone'] = location.get('timezone')
    weather_frames.append(frame)

if not weather_frames:
    raise ValueError('No hourly weather data found in the Open-Meteo raw response.')

df_wx = pd.concat(weather_frames, ignore_index=True)

print('shape:', df_wx.shape)
print('unique weather locations:',
      df_wx[['latitude', 'longitude']].drop_duplicates().shape[0])
df_wx.head()


shape: (2904, 30)
unique weather locations: 120


,time,temperature_850hPa,wind_speed_850hPa,wind_direction_850hPa,geopotential_height_850hPa,temperature_700hPa,wind_speed_700hPa,wind_direction_700hPa,geopotential_height_700hPa,temperature_500hPa,wind_speed_500hPa,wind_direction_500hPa,geopotential_height_500hPa,temperature_300hPa,wind_speed_300hPa,wind_direction_300hPa,geopotential_height_300hPa,temperature_250hPa,wind_speed_250hPa,wind_direction_250hPa,geopotential_height_250hPa,temperature_200hPa,wind_speed_200hPa,wind_direction_200hPa,geopotential_height_200hPa,latitude,longitude,elevation_m,location_id,timezone
0,2026-09-18T00:00,20.0,16.2,248,1484.0,6.4,46.1,222,3121.0,-7.5,111.8,237,5791.0,-35.5,111.3,240,9554.84,-43.0,107.5,241,10805.72,-51.5,121.1,231,12279.07,32.8125,35.125,10.0,0,GMT
1,2026-09-18T01:00,18.7,27.0,275,1483.0,6.5,55.4,217,3115.0,-6.8,107.5,241,5788.0,-35.5,109.1,240,9548.39,-43.5,106.6,239,10798.10,-51.0,120.5,232,12269.77,32.8125,35.125,10.0,0,GMT
2,2026-09-18T02:00,18.3,30.1,260,1482.0,6.1,51.5,214,3113.0,-6.8,106.3,243,5788.0,-35.5,105.8,241,9545.16,-44.0,105.6,238,10792.38,-51.0,116.9,234,12265.12,32.8125,35.125,10.0,0,GMT
3,2026-09-18T03:00,17.0,37.3,245,1484.0,5.6,53.0,218,3109.0,-6.5,98.3,246,5792.0,-35.5,94.0,242,9545.16,-43.5,101.1,240,10792.38,-51.5,107.0,240,12265.12,32.8125,35.125,10.0,0,GMT
4,2026-09-18T04:00,16.5,36.5,237,1487.0,5.6,57.9,224,3110.0,-6.5,99.6,248,5795.0,-35.5,83.2,245,9548.39,-43.5,96.6,243,10794.29,-52.0,96.6,245,12265.12,32.8125,35.125,10.0,0,GMT


In [11]:
print("Forecast objects returned:", len(meteo_locations))

coords = [
    (loc.get("latitude"), loc.get("longitude"))
    for loc in meteo_locations
]

print("Unique returned coordinates:", len(set(coords)))

from collections import Counter
duplicates = [coord for coord, count in Counter(coords).items() if count > 1]

print("Repeated coordinates:", duplicates)

Forecast objects returned: 121
Unique returned coordinates: 120
Repeated coordinates: [(26.81898, 49.933556)]


## 5. Open-Meteo — Profiling


In [12]:
profile_wx = profile(df_wx)
profile_wx


,column,dtype,null_count,percent_null,unique_count,sample_value
0,time,str,0,0.0,24,2026-09-18T00:00
1,temperature_850hPa,float64,0,0.0,126,20.0
2,wind_speed_850hPa,float64,0,0.0,462,16.2
3,wind_direction_850hPa,int64,0,0.0,201,248
4,geopotential_height_850hPa,float64,0,0.0,54,1484.0
5,temperature_700hPa,float64,0,0.0,79,6.4
6,wind_speed_700hPa,float64,0,0.0,650,46.1
7,wind_direction_700hPa,int64,0,0.0,293,222
8,geopotential_height_700hPa,float64,0,0.0,112,3121.0
9,temperature_500hPa,float64,0,0.0,28,-7.5


In [13]:
print('rows:', len(df_wx))
print('exact duplicate rows:', df_wx.duplicated().sum())
print('unique locations:', df_wx[['latitude', 'longitude']].drop_duplicates().shape[0])
print('duplicate (location, time) rows:',
      df_wx.duplicated(subset=['latitude', 'longitude', 'time']).sum())
print('missing timestamps:', df_wx['time'].isna().sum())

# Show all rows involved in duplicated location + time keys
weather_dup_mask = df_wx.duplicated(
    subset=['latitude', 'longitude', 'time'],
    keep=False
)

weather_duplicates = (
    df_wx[weather_dup_mask]
    .sort_values(['latitude', 'longitude', 'time'])
)

print("Rows involved in duplicates:", len(weather_duplicates))
print(
    "Duplicated location/time groups:",
    weather_duplicates.groupby(
        ['latitude', 'longitude', 'time']
    ).ngroups
)

weather_duplicates


rows: 2904
exact duplicate rows: 0
unique locations: 120
duplicate (location, time) rows: 24
missing timestamps: 0
Rows involved in duplicates: 48
Duplicated location/time groups: 24


,time,temperature_850hPa,wind_speed_850hPa,wind_direction_850hPa,geopotential_height_850hPa,temperature_700hPa,wind_speed_700hPa,wind_direction_700hPa,geopotential_height_700hPa,temperature_500hPa,wind_speed_500hPa,wind_direction_500hPa,geopotential_height_500hPa,temperature_300hPa,wind_speed_300hPa,wind_direction_300hPa,geopotential_height_300hPa,temperature_250hPa,wind_speed_250hPa,wind_direction_250hPa,geopotential_height_250hPa,temperature_200hPa,wind_speed_200hPa,wind_direction_200hPa,geopotential_height_200hPa,latitude,longitude,elevation_m,location_id,timezone
1056,2026-09-18T00:00,27.2,48.9,356,1498.0,16.0,17.5,6,3173.0,-3.5,10.3,144,5929.0,-30.0,14.5,246,9758.06,-40.5,30.0,233,11026.67,-51.5,41.3,216,12509.30,26.81898,49.933556,0.0,44,GMT
1944,2026-09-18T00:00,27.2,46.6,356,1499.0,16.0,20.5,360,3174.0,-3.5,9.7,150,5928.0,-30.0,16.1,243,9756.45,-40.5,31.5,230,11026.67,-51.5,42.0,217,12509.30,26.81898,49.933556,0.0,81,GMT
1057,2026-09-18T01:00,27.0,47.7,353,1500.0,16.0,16.8,2,3174.0,-3.5,11.1,139,5928.0,-30.0,16.1,243,9758.06,-40.5,30.7,231,11026.67,-51.5,40.6,214,12511.63,26.81898,49.933556,0.0,44,GMT
1945,2026-09-18T01:00,27.0,45.8,354,1501.0,15.8,21.1,358,3175.0,-3.5,9.7,150,5928.0,-30.0,16.7,240,9758.06,-40.5,32.3,228,11026.67,-51.5,42.0,217,12509.30,26.81898,49.933556,0.0,81,GMT
1058,2026-09-18T02:00,26.9,46.5,352,1499.0,15.8,18.6,2,3175.0,-3.7,11.1,139,5927.0,-30.0,15.0,241,9756.45,-40.5,31.5,230,11026.67,-51.5,39.4,218,12511.63,26.81898,49.933556,0.0,44,GMT
1946,2026-09-18T02:00,26.9,44.2,352,1500.0,15.7,23.6,360,3175.0,-3.5,11.3,148,5927.0,-30.0,16.7,240,9756.45,-40.5,33.2,229,11024.76,-51.5,40.9,220,12509.30,26.81898,49.933556,0.0,81,GMT
1059,2026-09-18T03:00,26.9,43.4,351,1501.0,15.7,19.3,6,3176.0,-4.0,12.0,143,5928.0,-30.0,15.6,238,9758.06,-40.5,32.3,228,11026.67,-51.5,39.2,220,12511.63,26.81898,49.933556,0.0,44,GMT
1947,2026-09-18T03:00,26.9,40.7,350,1501.0,15.7,24.9,3,3177.0,-3.7,12.4,151,5928.0,-30.0,17.3,236,9756.45,-40.5,33.1,226,11026.67,-51.5,40.0,221,12509.30,26.81898,49.933556,0.0,81,GMT
1060,2026-09-18T04:00,27.0,40.2,350,1505.0,15.7,18.9,9,3182.0,-4.0,13.0,146,5933.0,-30.0,15.4,231,9762.90,-40.5,32.2,225,11032.38,-52.0,38.2,224,12516.28,26.81898,49.933556,0.0,44,GMT
1948,2026-09-18T04:00,27.0,37.1,349,1506.0,15.5,25.6,6,3182.0,-4.0,14.5,156,5932.0,-30.0,18.0,233,9762.90,-40.5,34.8,224,11030.48,-52.0,39.0,225,12513.95,26.81898,49.933556,0.0,81,GMT


### Issues found — Open-Meteo

| # | Issue | Decision |
|---|---|---|
| 1 | New raw response contains multiple locations rather than the old single Riyadh point | Flatten every returned location |
| 2 | Hourly weather is stored as parallel arrays | Convert each location's `hourly` block to rows |
| 3 | `time` is text | Convert to UTC datetime |
| 4 | The same hour can appear at many locations, and repeated location/time keys may contain different weather values | Keep repeated (`latitude`, `longitude`, `time`) rows when weather values differ; remove only exact duplicate rows |
| 5 | Pressure-level variable names already encode the atmospheric level (`850hPa`, `700hPa`, etc.) | Keep them as separate columns for later aircraft-altitude matching |
| 6 | Missing weather values should not be guessed | Keep as missing; Task 3 validates them |


## 6. Open-Meteo — Cleaning


In [14]:
df_wx_clean = df_wx.copy()

# 1) Convert time to UTC and give it an analysis-friendly name.
df_wx_clean['time'] = pd.to_datetime(df_wx_clean['time'], utc=True, errors='coerce')
df_wx_clean = df_wx_clean.rename(columns={'time': 'observation_time'})

# 2) Ensure coordinates/elevation and weather measurements are numeric.
non_numeric_weather = {'observation_time', 'timezone'}
for col in df_wx_clean.columns:
    if col not in non_numeric_weather:
        df_wx_clean[col] = pd.to_numeric(df_wx_clean[col], errors='coerce')

# 3) Remove only true duplicates for the same location and hour.
# Do not drop rows only because returned latitude/longitude/time match.
# Some requested locations can map to the same Open-Meteo grid coordinates
# while retaining different location metadata.

exact_duplicates = df_wx_clean.duplicated().sum()

print("Exact duplicate weather rows:", exact_duplicates)

if exact_duplicates > 0:
    df_wx_clean = df_wx_clean.drop_duplicates()

print("Weather rows after cleaning:", len(df_wx_clean))


Exact duplicate weather rows: 0
Weather rows after cleaning: 2904


In [15]:
df_wx_clean.head()


,observation_time,temperature_850hPa,wind_speed_850hPa,wind_direction_850hPa,geopotential_height_850hPa,temperature_700hPa,wind_speed_700hPa,wind_direction_700hPa,geopotential_height_700hPa,temperature_500hPa,wind_speed_500hPa,wind_direction_500hPa,geopotential_height_500hPa,temperature_300hPa,wind_speed_300hPa,wind_direction_300hPa,geopotential_height_300hPa,temperature_250hPa,wind_speed_250hPa,wind_direction_250hPa,geopotential_height_250hPa,temperature_200hPa,wind_speed_200hPa,wind_direction_200hPa,geopotential_height_200hPa,latitude,longitude,elevation_m,location_id,timezone
0,2026-09-18 00:00:00+00:00,20.0,16.2,248,1484.0,6.4,46.1,222,3121.0,-7.5,111.8,237,5791.0,-35.5,111.3,240,9554.84,-43.0,107.5,241,10805.72,-51.5,121.1,231,12279.07,32.8125,35.125,10.0,0,GMT
1,2026-09-18 01:00:00+00:00,18.7,27.0,275,1483.0,6.5,55.4,217,3115.0,-6.8,107.5,241,5788.0,-35.5,109.1,240,9548.39,-43.5,106.6,239,10798.10,-51.0,120.5,232,12269.77,32.8125,35.125,10.0,0,GMT
2,2026-09-18 02:00:00+00:00,18.3,30.1,260,1482.0,6.1,51.5,214,3113.0,-6.8,106.3,243,5788.0,-35.5,105.8,241,9545.16,-44.0,105.6,238,10792.38,-51.0,116.9,234,12265.12,32.8125,35.125,10.0,0,GMT
3,2026-09-18 03:00:00+00:00,17.0,37.3,245,1484.0,5.6,53.0,218,3109.0,-6.5,98.3,246,5792.0,-35.5,94.0,242,9545.16,-43.5,101.1,240,10792.38,-51.5,107.0,240,12265.12,32.8125,35.125,10.0,0,GMT
4,2026-09-18 04:00:00+00:00,16.5,36.5,237,1487.0,5.6,57.9,224,3110.0,-6.5,99.6,248,5795.0,-35.5,83.2,245,9548.39,-43.5,96.6,243,10794.29,-52.0,96.6,245,12265.12,32.8125,35.125,10.0,0,GMT


In [16]:
weather_output = INTERIM_DIR / 'cleaned_weather.csv'
df_wx_clean.to_csv(weather_output, index=False)
print('saved:', weather_output.resolve())


saved: C:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\interim\cleaned_weather.csv


## 7. Aircraft Type Lookup — Load and Profile

The aircraft database is a reference dataset. Its `icao24` is renamed to `plane_id` so it can later join to OpenSky.

**Important for SkyPrint:** `typecode` will later determine whether an aircraft type can be mapped to the fuel/performance model.  
We therefore clean `plane_id` and `typecode` carefully before resolving duplicate `plane_id` records.


In [17]:
df_ac = pd.read_csv(RAW_DIR / 'aircraftDatabase.csv', dtype='string')
df_ac = df_ac.rename(columns={'icao24': 'plane_id'})

print('shape:', df_ac.shape)
df_ac.head(3)


shape: (520000, 27)


,plane_id,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,operatorcallsign,operatoricao,operatoriata,owner,testreg,registered,reguntil,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,false,false,false,<NA>,<NA>
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,<NA>,L1P,<NA>,<NA>,<NA>,<NA>,Vintage Aircraft Llc,<NA>,<NA>,2027-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,false,false,false,<NA>,<NA>
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,<NA>,L2P,<NA>,<NA>,<NA>,<NA>,Tvpx Aircraft Solutions Inc Trustee,<NA>,<NA>,2029-08-31,<NA>,1977-01-01,<NA>,<NA>,LYCOMING TI0-540 SER,false,false,false,<NA>,<NA>


In [18]:
profile_ac = profile(df_ac)
profile_ac


,column,dtype,null_count,percent_null,unique_count,sample_value
0,plane_id,string,1,0.00,519997,aa3487
1,registration,string,3474,0.67,514254,N757F
2,manufacturericao,string,91199,17.54,739,RAYTHEON
3,manufacturername,string,81434,15.66,41540,Raytheon Aircraft Company
4,model,string,79574,15.30,34482,A36
5,typecode,string,40053,7.70,1933,BE36
6,serialnumber,string,82882,15.94,295525,E-3121
7,linenumber,string,519029,99.81,693,AF1085
8,icaoaircrafttype,string,91216,17.54,51,L1P
9,operator,string,496302,95.44,4001,Indian Air Force


In [19]:
null_pct = (df_ac.isna().sum() / len(df_ac) * 100).round(2).sort_values(ascending=False)

print('rows with plane_id null:', df_ac['plane_id'].isna().sum())
print('exact duplicate rows:', df_ac.duplicated().sum())
print('duplicate plane_id rows (before normalization):',
      df_ac[df_ac['plane_id'].notna()].duplicated(subset=['plane_id']).sum())
print('typecode nulls:', df_ac['typecode'].isna().sum())
print()
print('columns with >= 90% nulls:')
print(null_pct[null_pct >= 90])


rows with plane_id null: 1
exact duplicate rows: 2
duplicate plane_id rows (before normalization): 2
typecode nulls: 40053

columns with >= 90% nulls:
status                 100.00
seatconfiguration      100.00
firstflightdate         99.94
testreg                 99.94
notes                   99.93
linenumber              99.81
categoryDescription     98.61
operatoriata            98.45
operator                95.44
operatorcallsign        92.29
operatoricao            92.04
dtype: float64


### Issues found — Aircraft Database

| # | Issue | Decision |
|---|---|---|
| 1 | Rows without `plane_id` cannot join to OpenSky | Drop |
| 2 | `plane_id` may differ only by case/whitespace | Trim + lowercase before duplicate checks |
| 3 | `typecode` may differ only by case/whitespace (for example `b738`, `B738 `) | Trim + uppercase before comparing |
| 4 | Blank-like text can look like a real value | Convert empty/`nan`/`none`/`null` text to missing |
| 5 | Duplicate `plane_id` rows may contain the same useful `typecode` plus missing values | Treat as one aircraft; keep the one normalized non-null typecode |
| 6 | Duplicate `plane_id` rows may contain genuinely different non-null `typecode`s | Do **not** guess; set `typecode` missing and `typecode_status = conflict` |
| 7 | A `plane_id` with no usable `typecode` | Keep aircraft row with `typecode_status = missing` |
| 8 | Some columns are zero-variance or ≥90% null and were already out of project scope | Keep the team's existing fixed drop list |
| 9 | `built`, `registered`, `reguntil` are date text | Convert to datetime |


## 8. Aircraft Type Lookup — Cleaning


In [20]:
df_ac_clean = df_ac.copy()

# 1) Normalize missing plane IDs, then remove rows that cannot join to OpenSky.
df_ac_clean['plane_id'] = (
    df_ac_clean['plane_id']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({'': pd.NA, 'nan': pd.NA, 'none': pd.NA, 'null': pd.NA})
)
df_ac_clean = df_ac_clean.dropna(subset=['plane_id'])

# 2) Normalize typecode BEFORE deciding whether duplicate plane IDs conflict.
# We intentionally normalize case/whitespace only; we do not remove punctuation blindly.
df_ac_clean['typecode'] = (
    df_ac_clean['typecode']
    .astype('string')
    .str.strip()
    .str.upper()
    .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, 'NULL': pd.NA})
)



# 3) Drop exact duplicates after normalization.
before = len(df_ac_clean)
df_ac_clean = df_ac_clean.drop_duplicates()
print(f'dropped {before - len(df_ac_clean)} exact duplicate aircraft rows after normalization')

# 4) Drop columns that are clearly unusable/out of scope for this project.
cols_to_drop = [
    'modes', 'adsb', 'acars',
    'status', 'seatconfiguration', 'firstflightdate',
    'testreg', 'notes', 'linenumber',
    'operatoriata', 'operatorcallsign'
]

cols_to_drop = [c for c in cols_to_drop if c in df_ac_clean.columns]
df_ac_clean = df_ac_clean.drop(columns=cols_to_drop)

# 5) Parse date-like text columns that survived.
for col in ['built', 'registered', 'reguntil']:
    if col in df_ac_clean.columns:
        df_ac_clean[col] = pd.to_datetime(df_ac_clean[col], errors='coerce')


dropped 2 exact duplicate aircraft rows after normalization


In [21]:
# Inspect duplicate plane_ids AFTER normalization and count distinct usable typecodes.
typecodes_per_plane = (
    df_ac_clean.groupby('plane_id')['typecode']
    .agg(lambda s: sorted(set(s.dropna())))
)

duplicate_plane_ids = df_ac_clean.loc[
    df_ac_clean.duplicated('plane_id', keep=False), 'plane_id'
].unique()

print('duplicate plane_ids after normalization:', len(duplicate_plane_ids))

duplicate_typecode_summary = pd.DataFrame({
    'plane_id': typecodes_per_plane.index,
    'unique_non_null_typecodes': typecodes_per_plane.values
})
duplicate_typecode_summary['typecode_count'] = (
    duplicate_typecode_summary['unique_non_null_typecodes'].str.len()
)
duplicate_typecode_summary = duplicate_typecode_summary[
    duplicate_typecode_summary['plane_id'].isin(duplicate_plane_ids)
]

duplicate_typecode_summary.head(20)


duplicate plane_ids after normalization: 6


,plane_id,unique_non_null_typecodes,typecode_count
124564,71010b,[A342],1
124640,7101e7,[GLF4],1
124641,7101e9,[GLF4],1
124644,7101f0,[GLF4],1
124664,71022b,[B739],1
124807,71032c,[B752],1


In [22]:
# Resolve only duplicated plane_ids — much faster than looping over all aircraft

duplicate_mask = df_ac_clean.duplicated(subset=['plane_id'], keep=False)

df_unique = df_ac_clean[~duplicate_mask].copy()
df_duplicates = df_ac_clean[duplicate_mask].copy()

# Normal rows
df_unique['typecode_status'] = np.where(
    df_unique['typecode'].notna(),
    'matched',
    'missing'
)
df_unique['source_rows'] = 1
df_unique['typecode_candidates'] = df_unique['typecode']


# Resolve only the duplicated plane_ids
resolved_duplicates = []

for plane_id, group in df_duplicates.groupby('plane_id'):

    valid_typecodes = sorted(set(group['typecode'].dropna()))

    # Keep the row with the most available metadata
    result = group.loc[group.notna().sum(axis=1).idxmax()].copy()

    if len(valid_typecodes) == 0:
        result['typecode'] = pd.NA
        result['typecode_status'] = 'missing'

    elif len(valid_typecodes) == 1:
        result['typecode'] = valid_typecodes[0]
        result['typecode_status'] = 'matched'

    else:
        result['typecode'] = pd.NA
        result['typecode_status'] = 'conflict'

    result['source_rows'] = len(group)
    result['typecode_candidates'] = (
        '|'.join(valid_typecodes) if valid_typecodes else pd.NA
    )

    resolved_duplicates.append(result)


df_resolved_duplicates = pd.DataFrame(resolved_duplicates)

# Put unique + resolved duplicated aircraft back together
df_ac_clean = pd.concat(
    [df_unique, df_resolved_duplicates],
    ignore_index=True
)

print('final aircraft rows:', len(df_ac_clean))
print('unique plane_ids:', df_ac_clean['plane_id'].nunique())

print('\ntypecode status:')
print(df_ac_clean['typecode_status'].value_counts())

print('\nremaining duplicate plane_ids:',
      df_ac_clean.duplicated(subset=['plane_id']).sum())

final aircraft rows: 519991
unique plane_ids: 519991

typecode status:
typecode_status
matched    479945
missing     40046
Name: count, dtype: int64

remaining duplicate plane_ids: 0


In [23]:
# Show any real conflicts for review. They remain in the dataset, but typecode is not guessed.
conflicts = df_ac_clean[df_ac_clean['typecode_status'] == 'conflict'][
    ['plane_id', 'typecode', 'typecode_candidates', 'source_rows']
]
print('real typecode conflicts:', len(conflicts))
conflicts.head(20)


real typecode conflicts: 0


,plane_id,typecode,typecode_candidates,source_rows


In [24]:
df_ac_clean.head()


,plane_id,registration,manufacturericao,manufacturername,model,typecode,serialnumber,icaoaircrafttype,operator,operatoricao,owner,registered,reguntil,built,engines,categoryDescription,typecode_status,source_rows,typecode_candidates
0,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,L1P,<NA>,<NA>,Vintage Aircraft Llc,NaT,2027-01-31,NaT,<NA>,<NA>,matched,1,BE36
1,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,L2P,<NA>,<NA>,Tvpx Aircraft Solutions Inc Trustee,NaT,2029-08-31,1977-01-01,LYCOMING TI0-540 SER,<NA>,matched,1,PA31
2,a7a809,N5926K,ROCKWELL,<NA>,<NA>,AC90,<NA>,L2T,<NA>,<NA>,<NA>,NaT,NaT,NaT,<NA>,<NA>,matched,1,AC90
3,391927,F-GGJH,ROBIN,Robin,DR.400 160 Chevalier,DR40,1795,L1P,<NA>,<NA>,Private,NaT,NaT,NaT,<NA>,<NA>,matched,1,DR40
4,503c21,LY-KNA,<NA>,Impulse Aircraft,Impulse 100,ZZZZ,<NA>,<NA>,<NA>,<NA>,Private,NaT,NaT,NaT,<NA>,<NA>,matched,1,ZZZZ


In [25]:
# Final check for issues 8 and 9

print("Columns after cleaning:")
print(df_ac_clean.columns.tolist())

print("\nDate column types:")
for col in ['built', 'registered', 'reguntil']:
    print(col, ":", df_ac_clean[col].dtype)

Columns after cleaning:
['plane_id', 'registration', 'manufacturericao', 'manufacturername', 'model', 'typecode', 'serialnumber', 'icaoaircrafttype', 'operator', 'operatoricao', 'owner', 'registered', 'reguntil', 'built', 'engines', 'categoryDescription', 'typecode_status', 'source_rows', 'typecode_candidates']

Date column types:
built : datetime64[us]
registered : datetime64[us]
reguntil : datetime64[us]


In [26]:
aircraft_output = INTERIM_DIR / 'cleaned_aircraft.csv'
df_ac_clean.to_csv(aircraft_output, index=False)
print('saved:', aircraft_output.resolve())


saved: C:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\interim\cleaned_aircraft.csv


## 9. Summary


In [27]:
summary = pd.DataFrame([
    {
        'dataset': 'OpenSky states',
        'raw_rows': len(df_sky),
        'cleaned_rows': len(df_sky_clean),
        'output': 'data/interim/cleaned_opensky.csv'
    },
    {
        'dataset': 'Open-Meteo hourly by location',
        'raw_rows': len(df_wx),
        'cleaned_rows': len(df_wx_clean),
        'output': 'data/interim/cleaned_weather.csv'
    },
    {
        'dataset': 'Aircraft type lookup',
        'raw_rows': len(df_ac),
        'cleaned_rows': len(df_ac_clean),
        'output': 'data/interim/cleaned_aircraft.csv'
    }
])

summary


,dataset,raw_rows,cleaned_rows,output
0,OpenSky states,131,131,data/interim/cleaned_opensky.csv
1,Open-Meteo hourly by location,2904,2904,data/interim/cleaned_weather.csv
2,Aircraft type lookup,520000,519991,data/interim/cleaned_aircraft.csv


In [28]:
print(df_ac_clean.columns.tolist())

['plane_id', 'registration', 'manufacturericao', 'manufacturername', 'model', 'typecode', 'serialnumber', 'icaoaircrafttype', 'operator', 'operatoricao', 'owner', 'registered', 'reguntil', 'built', 'engines', 'categoryDescription', 'typecode_status', 'source_rows', 'typecode_candidates']


In [29]:

print("=== FINAL CLEANING CHECK ===")
print("----------------------------")

# OpenSky
print("OpenSky rows:", len(df_sky_clean))
print(
    "OpenSky duplicate keys:",
    df_sky_clean.duplicated(
        ['plane_id', 'time_position']
    ).sum()
)

# Weather
print("\nWeather rows:", len(df_wx_clean))
print(
    "Weather location IDs:",
    df_wx_clean['location_id'].nunique()
)
print(
    "Weather duplicate keys:",
    df_wx_clean.duplicated(
        ['location_id', 'observation_time']
    ).sum()
)

# Aircraft
print("\nAircraft rows:", len(df_ac_clean))
print(
    "Aircraft duplicate IDs:",
    df_ac_clean['plane_id'].duplicated().sum()
)
print(
    "Aircraft typecode conflicts:",
    (df_ac_clean['typecode_status'] == 'conflict').sum()
)

print("\nTypecode status:")
print(
    df_ac_clean['typecode_status']
    .value_counts(dropna=False)
)

=== FINAL CLEANING CHECK ===
----------------------------
OpenSky rows: 131
OpenSky duplicate keys: 0

Weather rows: 2904
Weather location IDs: 121
Weather duplicate keys: 0

Aircraft rows: 519991
Aircraft duplicate IDs: 0
Aircraft typecode conflicts: 0

Typecode status:
typecode_status
matched    479945
missing     40046
Name: count, dtype: int64
